# SignSense AI — BiLSTM Training (Kaggle)

**Model:** Bidirectional LSTM sequence classifier (word/phrase mode)  
**Target accuracy:** > 85% | **Runtime:** ~45 min on Kaggle T4 GPU

### Before you start
1. Settings → Accelerator → **GPU T4 x2**
2. Add dataset: **+ Add Data** → search `grassknoted/asl-alphabet` → Add
3. Run `kaggle_train_mlp.ipynb` first OR add the preprocessed .npy files as a dataset
4. Run all cells top to bottom

### Note on training data
LSTM uses synthetic sequences (tile + jitter from static frames). Sufficient for a working model.

In [ ]:
# ── Cell 1: Setup paths ───────────────────────────────────────────────────────
import os

WORKING_DIR   = '/kaggle/working'
MODELS_DIR    = f'{WORKING_DIR}/models'
LOGS_DIR      = f'{WORKING_DIR}/logs/lstm'
KAGGLE_INPUT  = '/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train'

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR,   exist_ok=True)

print(f'Models will be saved to: {MODELS_DIR}')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q 'protobuf>=5.28.0' 'mediapipe>=0.10.18' scikit-learn tqdm
print('✅ Dependencies installed.')

In [ ]:
# ── Cell 3: Clone project repo from GitHub ────────────────────────────────────
import os, sys

REPO_URL = 'https://github.com/prateek1756/sign-language-detection.git'

if not os.path.exists(f'{WORKING_DIR}/sign-language-detection'):
    !git clone {REPO_URL} {WORKING_DIR}/sign-language-detection
else:
    !git -C {WORKING_DIR}/sign-language-detection pull

BACKEND_PATH = f'{WORKING_DIR}/sign-language-detection/backend'
sys.path.insert(0, BACKEND_PATH)

from configs.training_config import ASL_CLASSES, NUM_CLASSES
print(f'✅ Repo ready — {NUM_CLASSES} classes')

In [ ]:
# ── Cell 4: Load preprocessed data ───────────────────────────────────────────
import os, shutil, numpy as np
from pathlib import Path

RAW_ASL_DIR   = f'{BACKEND_PATH}/data/raw/ASL'
PROCESSED_DIR = f'{BACKEND_PATH}/data/processed/ASL'
PROCESSED_NPY = f'{PROCESSED_DIR}/landmarks_all.npy'
LABELS_NPY    = f'{PROCESSED_DIR}/labels_all.npy'

if os.path.exists(PROCESSED_NPY):
    print('✅ Preprocessed data found — skipping preprocessing.')
else:
    # Check if preprocessed data was saved as a Kaggle dataset
    # (upload landmarks_all.npy + labels_all.npy as a private dataset)
    npy_input = '/kaggle/input/signsense-preprocessed/landmarks_all.npy'
    if os.path.exists(npy_input):
        print('Restoring preprocessed data from Kaggle dataset...')
        os.makedirs(PROCESSED_DIR, exist_ok=True)
        for f in ['landmarks_all.npy', 'labels_all.npy', 'class_map.json']:
            src = f'/kaggle/input/signsense-preprocessed/{f}'
            if os.path.exists(src):
                shutil.copy(src, f'{PROCESSED_DIR}/{f}')
                print(f'  ✅ {f}')
    else:
        # Re-preprocess from the ASL dataset
        if not os.path.exists(KAGGLE_INPUT):
            raise FileNotFoundError('Add grassknoted/asl-alphabet dataset first.')
        print('Copying dataset and preprocessing (~15 min)...')
        os.makedirs(RAW_ASL_DIR, exist_ok=True)
        for class_dir in Path(KAGGLE_INPUT).iterdir():
            if class_dir.is_dir():
                dest = Path(RAW_ASL_DIR) / class_dir.name
                if not dest.exists():
                    shutil.copytree(str(class_dir), str(dest))
        !python {BACKEND_PATH}/src/preprocess.py --all --augment --aug_factor 3

X = np.load(PROCESSED_NPY)
y = np.load(LABELS_NPY)
print(f'\n✅ Data ready: X={X.shape}  y={y.shape}  classes={len(set(y.tolist()))}')

In [ ]:
# ── Cell 5: Verify GPU ────────────────────────────────────────────────────────
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('✅ GPU memory growth enabled.')
else:
    print('WARNING: No GPU. LSTM training will be slow (~3-4h).')

In [ ]:
# ── Cell 6: Train BiLSTM ──────────────────────────────────────────────────────
import sys
from pathlib import Path

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['train', 'training_config', 'configs']):
        del sys.modules[mod]

from configs.training_config import LSTMConfig
from src.train import train_lstm

cfg = LSTMConfig()
cfg.save_dir = Path(MODELS_DIR)
cfg.log_dir  = Path(LOGS_DIR)
cfg.mixed_precision = False

print(f'  lstm_units:    {cfg.lstm_units}')
print(f'  sequence_len:  {cfg.sequence_len}')
print(f'  epochs:        {cfg.epochs}')
print(f'  batch_size:    {cfg.batch_size}')
print()

model = train_lstm(cfg)
print('\n✅ Training complete!')

In [ ]:
# ── Cell 7: Evaluate ──────────────────────────────────────────────────────────
import sys
from pathlib import Path

for mod in list(sys.modules.keys()):
    if 'evaluate' in mod:
        del sys.modules[mod]

import src.evaluate as ev
ev.MODELS_DIR = Path(MODELS_DIR)

results = ev.evaluate('asl_lstm', split='test')
print(f"\nFinal test accuracy: {results['accuracy']*100:.1f}%")
print(f"Top-5 accuracy:      {results['top5']*100:.1f}%")

In [ ]:
# ── Cell 8: Verify saved model ────────────────────────────────────────────────
import os, numpy as np, tensorflow as tf
from configs.training_config import LSTMConfig

print('Files saved:')
for f in sorted(os.listdir(MODELS_DIR)):
    size_mb = os.path.getsize(os.path.join(MODELS_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

cfg    = LSTMConfig()
loaded = tf.keras.models.load_model(os.path.join(MODELS_DIR, 'asl_lstm.keras'))
dummy  = np.zeros((1, cfg.sequence_len, 63), dtype=np.float32)
pred   = loaded.predict(dummy, verbose=0)
print(f'\nSanity check — output shape: {pred.shape}  sum: {pred.sum():.4f}')
print('\n✅ Model verified. Download asl_lstm.keras from the Output tab.')